# Exercise 02 Workbook: your own protein

This notebook is **deliberately almost empty**, and it is entirely about **your** protein: the
one on your ex01 Character Sheet, which you carry through ex03, ex04 and your final project.

Its companion, **`ex02_guide.ipynb`**, works everything through on human p53 and PDB 3D08.
**Read the guide first**, then work here with it open beside you. The sections below carry the
same numbers as the guide's: A.1 to A.3, then B.1 to B.3.

## How to work in this notebook

- **Copy code from the guide and adapt it.** Its helpers work on any protein with an
  AlphaFold DB entry.
- **Structure the notebook however you like.** The sections are a checklist, not a template.
- **The reasoning is graded, not the code.** Every section asks for a written conclusion.
- **No GPU here.** B.1's AlphaFold 3 run happens in ColabFold's notebook, on
  Colab's free GPU.

**Time: about 45 to 60 minutes**, plus the wait for AlphaFold 3. **What you will not learn
here** is anything the guide did not show: every section asks for a guide step, repeated on
your protein.

## Using AI tools

You may use AI assistants for syntax, debugging and code snippets. You must supply the
predictions, the critical evaluation and the biological reasoning. **"The AI said so" is not
an answer.**

**Predict-first prompts** (A.1 and B.1) ask for your prediction *before* you run
anything. That written prediction is the graded artefact. Being wrong costs nothing; not
predicting costs the marks.

## Setup

Copy the guide's setup cell here and run it first. A Colab notebook starts empty, so
nothing from the guide's session carries over here.

In [1]:
#@title Setup {display-mode: "form"}
# Setup. Everything in this notebook runs on a plain CPU runtime.
try:
    from google.colab import drive  # noqa: F401  (import is the availability test)

    is_google_colab = True
except ImportError:
    is_google_colab = False

if is_google_colab:
    %pip install numpy==2.1.3 pandas==2.2.3 matplotlib==3.10.0 py3dmol==2.4.0 mdanalysis==2.10.0 biopython==1.85 rdkit==2026.3.5

import warnings

warnings.filterwarnings("ignore")

import json
import os
import re
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import pandas as pd
import py3Dmol
import requests
from Bio.PDB import PDBIO, MMCIFParser
from MDAnalysis.analysis import align, distances
from rdkit import Chem
from rdkit.Chem import rdMolAlign

print("Libraries loaded -- no GPU needed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.9 MB/s eta 0:00:00


Libraries loaded -- no GPU needed


---

## Your structure: the crystal from ex01

This notebook works on **the crystal you chose in ex01**, the PDB entry on your Character
Sheet, and on no other: not another entry of the same protein, even a sharper one. The
residue numbers you carry from one exercise to the next belong to that entry. Every
candidate numbers its residues the way UniProt does, so they also line up with both AlphaFold models, as long as B.1 gets your whole
UniProt sequence.

Put its PDB ID at the top of your first code cell, as `PDB_ID = "..."`, and use `PDB_ID`
wherever the guide names its own entry.

---

# Part A: Your protein's AlphaFold prediction

## A.1 Predict first, before you fetch anything

**Before you look at any number**, predict: will confidence be uniform or split, and where?
Base it on your ex01 findings: the residues your crystal failed to resolve, and those with the
highest B-factors. A wrong prediction with stated reasoning earns full marks; "I don't know"
earns none.

Protein missing from [the AlphaFold Database](https://alphafold.ebi.ac.uk/)? Use the
[AlphaFold Server](https://alphafoldserver.com/welcome) or
[ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb).

> **My prediction:** The AlphaFold prediction is not uniform.
>
> **My reasoning (citing my ex01 findings):** 1STD has the X-ray diffraction solved for a 162-aa long subunit while the deposited residue count is 172. The first 8 amino acid at amino terminus and the last 2 at carboxyl terminus are not modelled due to the lack of electron density. I think the AlphaFold prediction will return low confidence score for these two regions. Though the amino acid with high B-factor means that the atom motion is dynamic, the electron density is still there so I guess the AlphaFold is able to predict its structure with good confidence.

## A.2 Confidence analysis

Produce, for your protein: the global mean pLDDT, a per-residue **pLDDT plot**, a **PAE
heatmap** and the four `fractionPlddt*` bands.

**Then answer:**

- Where is pLDDT above 90, and where below 70? Why there: core, surface, loops, termini?
- Which parts does the PAE place confidently relative to each other, and which not?
- Is your protein uniformly mediocre, or excellent in one region and uncertain elsewhere?
- How did your A.1 prediction hold up? If it was wrong, what did you miss?

## A.3 The AlphaFold model against your crystal

Superimpose the AlphaFold DB model on your ex01 crystal (`PDB_ID`) as the guide does. Report the
overall Cα RMSD, over how many matched residues, and the highest-deviation regions.

**Then answer:** why do those regions differ? An unresolved crystal loop nearby, a mutation, a
bound ligand or ion the model lacks, or a genuinely wrong prediction?

If your protein has no AlphaFold DB entry, **say so** rather than forcing the comparison. That
is a real outcome, not a failure.

---

# Part B: AlphaFold 3, with your ligand

## B.1 Predict, then run AlphaFold 3

**Before you run anything**, predict: where will AlphaFold 3 put your ex01 ligand, and how
sure will it be? Base it on your ex01 contact residues and your A.3 deviations near them.

Then run AlphaFold 3 in ColabFold's notebook as the guide's B.1 does, with your whole UniProt
sequence and your ligand's CCD code.

> **My prediction:**
>
> **My reasoning (citing my ex01 contacts and A.3):**

## B.2 What AlphaFold 3 returns

Upload your ZIP and unpack it as the guide does. Report `ranking_score`, `ptm`, the
protein-ligand `chain_pair_iptm` and `chain_pair_pae_min`, your ligand's mean pLDDT, and
AlphaFold 3's per-residue pLDDT drawn over your A.2 curve.

**Then answer:** which of these numbers is about your ligand, and what does each say? Where
do the two pLDDT curves part, and why might they?

## B.3 Your ligand against your crystal

Superpose the AlphaFold 3 model on your crystal (`PDB_ID`) as the guide's B.3 does. Report
the Cα RMSD, your ligand's RMSD against the crystal pose with how many atoms matched, and the
residues within 4 Å of each pose, as in ex01.

**Then answer:** was your B.1 prediction right? AlphaFold 3 trained on PDB entries released
before 30 September 2021: was yours one? Would a small ligand RMSD prove that your ligand
binds there? What would?

---

## Before you submit

- [ ] A.1 prediction **written before** the analysis in A.2, citing your ex01 findings
- [ ] pLDDT plot, PAE heatmap and confidence **composition** for your own protein
- [ ] A.3 comparison done, **or** explicitly explained why it could not be
- [ ] B.1 prediction **written before** your AlphaFold 3 run
- [ ] B.2 AlphaFold 3's confidence reported, with the numbers that concern your ligand
- [ ] B.3 ligand RMSD against your crystal, with the residues around both poses
- [ ] Anything that did not work is **described honestly** rather than deleted: a documented
      dead end earns marks, a silently missing section does not